# Plotting notebook for PiC_UVnudge runs
## Set up
### Packages

In [ ]:
import numpy as np
import xarray as xr
import xesmf as xe
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter
import pandas as pd
import scipy
from scipy import stats
import matplotlib as mpl
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.mathtext import _mathtext as mathtext
import matplotlib.ticker as mticker
from matplotlib import font_manager
from matplotlib import gridspec, animation
import matplotlib.path as mpath
import matplotlib.colors as colors
import matplotlib.dates as mdates
from matplotlib.patches import Ellipse
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.util import add_cyclic_point
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import warnings
warnings.simplefilter('ignore', UserWarning)
warnings.filterwarnings('ignore')
import datetime as dt
from datetime import timedelta
from cmcrameri import cm
import jinja2
from Plotting_functions import arclats, lons, CustomCmap, draw_circle
import cftime
import dask
from dask_jobqueue import PBSCluster
from dask.distributed import Client
from functools import partial
from collections import defaultdict
from pathlib import Path
import itertools
import sys
import string

In [2]:
font_path = '/glade/work/glydia/conda-envs/Helvetica.ttf' 
boldfont_path = '/glade/work/glydia/conda-envs/Helvetica-Bold.ttf' 
font_manager.fontManager.addfont(font_path)
font_manager.fontManager.addfont(boldfont_path)

mpl.rcParams['font.family'] = 'sans-serif'
mpl.rcParams['font.sans-serif'] = 'Helvetica'

### Filepaths and name variables

In [ ]:
## Plot types to make - CHANGE
# 0: Weighted spatial mean or Arctic sea ice area
# 1: Spatial maximum (AMOC)
# 2: Leave alone (doing spatial map or sea ice concentration)
plots = {
    'mtrd': [False, 0],  # Plot monthly and annual trends
    'comb': [True, 0], # Plot linear and monthly/annual trends together 
}

## Categorical plot type - DO NOT CHANGE
plot_types = {
    'line': False,
    'mtrd': False
}

# Set up plot_types based on plots
for pl, att in plots.items():
    if att[0]:
        if pl == 'mtrd':
            plot_types['mtrd'] = True
        elif att[1] <= 1:
            plot_types['line'] = True

# Spatial & time domain - CHANGE s_domain & t_domain only
s_domain = 1 # 0: Global, 1: Arctic, 2: Mid-latitudies
r_domain = False # True: use Arctic regional domain, False: pan-Arctic average
a_domain = False # True: 50-90, False: 70-90
o_domain =  True # True: 20-60N max at each lat, False: 33-55N max
t_domain = 1950 # start year
trd_length = 20 # Length of running trends - used for mtrd plots only
remove_amoc = False # Remove influence of AMOC

## Time averaging type - CHANGE
time_avg = 4   # 0: Monthly, 1: Yearly, 2: Seasonal, 3: All data, 4: Timeseries

## Ensemble mean or All members - CHANGE
ens_type = 0  # 0: All_members, 1: Mean

## Variables - CHANGE
# Primary variable
comp = 'ice'
freq = 0 # 0: monthly, 1: daily
var_ind = 0

# Secondary variable
comp2 = 'ocn'
var_ind2 = 0

# CHANGE
type_onemonth = 9 # Selects single month to plot timeseries for (time_avg = 4), if doing all months, equals 0
ocn_lat = 26.5
trend_months = [3,6,9,12]

In [ ]:
# DO NOT CHANGE
var_list = {'atm': ['TREFHT'],
            'ice': ['aice','hi'],
            'ocn': ['MOC']}
var_ext = {0: '', 1: '_d'}
var = var_list[comp][var_ind]+var_ext[freq]
var2 = var_list[comp2][var_ind2]+var_ext[freq]

In [ ]:
## Test numbers - DO NOT CHANGE
tst_nums = np.arange(1,4)

## Test names - True means plot, False means don't plot
# CHANGE
ds_plot_list = {
    'PiC_UVnudge_LM2006': True,
    'HIST_UVnudge_LM': True,
    'GHG_UVnudge_LM': True,
    'LENS2 piControl': True,
    'LENS2 HIST-SSP370': True,
    'GHG2': True,
    # 'GISTEMP': False,
    # 'HadCRUT5': False,
    # 'BEST': False,
    # 'NSIDC': False,
    # 'OSISAF': False,
    # 'ERA5': False,
    # 'RAPID': False,
}

## Filepaths - DO NOT CHANGE
path_to_work = '/glade/work/glydia/'
path_to_data = path_to_work+'Arctic_controls_processed_data/recent_plotting_data/'
path_to_graphs = '/glade/u/home/glydia/Recent_PiC_HIST_nudge_graphs/'

# Conditions - DO NOT CHANGE
vert_lev = {'atm': [False],
            'ice': [False,False],
            'ocn': [False]}
file_bool = not vert_lev[comp][var_ind] and freq == 0

In [6]:
########################## DO NOT CHANGE ANYTHING BELOW THIS LINE #############################

In [ ]:
%%time
    
## Select plot type
time_str_list = {0: 'month', 1: 'year', 2: 'season', 3: 'all', 4: 'timeseries'}
time_outstr = time_str_list[time_avg]

## Select ensemble type
ens_str_list = {0: 'All_members', 1: 'Mean'}
ens_str = ens_str_list[ens_type]

## Select time and spatial domain strings
sd_str_list = {0: 'Global', 1: 'Arctic'}
s_domain = 0 if (var == 'MOC') else s_domain # Make sure MOC is global
sd_str = sd_str_list[s_domain]

rd_str_list = {True: 'Reg', False: ''}
rd_str = rd_str_list[r_domain]

od_str_list = {True: 'Lat', False: ''}
od_str = od_str_list[o_domain]

td_str = str(t_domain)
t_end = 2024

month_str = np.array(['January','February','March','April','May','June','July','August','September','October','November','December'])
mon_str = np.array([i[0:3] for i in month_str])
m_str = np.array([i[0] for i in month_str])
seas_str = np.array(['MAM','JJA','SON','DJF'])
mon_seas_dict = {3: 'MAM',
                 6: 'JJA',
                 9: 'SON',
                12: 'DJF'}

CPU times: user 65 µs, sys: 0 ns, total: 65 µs
Wall time: 71 µs


In [ ]:
## Set up properties of each dataset and whether it will be plotted or not
# O: LENS ensemble
# 1: PiC_UVnudge ensemble
# 2: PiC_UVnudge single run
# 3: observations
# attribute structure: [use dataset[0], dataset typ[1], line color[2], line style[3], zorder[4], note[5](optional)]
use_era5 = var in ['aice','TREFHT','Z3']
use_ceres = var in ['RESTOM','FSNTOA','TOT_CLD_VISTAU']
ds_names = {
    'ERA5': [use_era5, 3, 'black', '-', 10, 'era5'],
    'GISTEMP': [var == 'TREFHT' and (plot_types['mtrd'] or plot_types['line']), 3, 'black', '-', 10 , 'gist'],
    'HadCRUT5': [var == 'TREFHT' and (plot_types['mtrd'] or plots['comb'][0]), 3, 'black', '-', 10 , 'had5'],
    'BEST': [var == 'TREFHT' and (plot_types['mtrd'] or plots['comb'][0]), 3, 'black', '-', 10 , 'best'],
    'NSIDC': [var == 'aice' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'nsii'],
    'OSISAF': [var == 'aice' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'osii'],
    'RAPID': [var == 'MOC' and ((plot_types['mtrd'] and trd_length != 40) or plot_types['line']), 3, 'black', '-', 10, 'rap'],
    'LENS2 piControl': [True, 0, 'tab:blue', '-', 1],
    'LENS2 HIST-SSP370': [True, 0, 'tab:red', '-', 1],
    'GHG2': [True, 0, 'tab:green', '-', 1],
    'PiC_UVnudge_LM2006': [None, 1, 'blue', '-', 4, 'lm06'],
    'HIST_UVnudge_LM': [None, 1, 'red', '-', 5, 'lmh'],
    'GHG_UVnudge_LM': [None, 1, 'lime', '-', 5, 'lmghg'],
}

ds_attr_names = {'Non-GHG': [2, 'tab:orange'],
             'Greenhouse gases': [1, 'tab:green'],
             'Winds': [3, 'tab:blue'],
             'HIST+winds': [0, 'tab:red']}

# ds_2order_names = 

ds_paper_names = {
    'LENS2 HIST-SSP370': 'HIST-SSP370-control',
    'GHG2': 'GHG-control',
    'LENS2 piControl': 'PI-control',
    'PiC_UVnudge_LM2006': 'PI+winds',
    'ERA5': 'OBS (ERA5)',
    'GISTEMP': 'OBS (GISTEMP)',
    'HadCRUT5': 'OBS (HadCRUT5)',
    'BEST': 'OBS (BEST)',
    'NSIDC': 'OBS (NSIDC)',
    'OSISAF': 'OBS (OSISAF)',
    'RAPID': 'OBS (RAPID)',
    'HIST_UVnudge_LM': 'HIST+winds',
    'GHG_UVnudge_LM': 'GHG+winds',
}

region_names = {
    'HA': 'High Arctic',
    'G': 'Greenland',
    'AA': 'Atlantic Arctic',
    'WS': 'Western Siberia',
    'ES': 'Eastern Siberia',
    'PA': 'Pacific Arctic'
}

# Update PiC_UVnudge 'use dataset' values with T/F from ds_plot_list
for dsname, use in ds_plot_list.items():
    ds_names[dsname][0] = use
    if dsname == 'ERA5':
        use_era5 = use

In [9]:
## Determine note type based on model runs plotted
note = ''
driftnote = ''

for dsname, attrs in ds_names.items():
    # Only pick datasets that will plotted and that have a code
    if attrs[0] and len(attrs) == 6:
        # Each PiC_UVnudge run has its own code, if the run will be plotted, it's code will be added to the note
        note = note+attrs[5]+'-'

note = 'era5' if note == '' else note[:-1]
driftnote = driftnote if driftnote == '' else driftnote[:-1]

### Custom functions
#### LoadData

In [10]:
def LoadData(plot_type, varname, tavg, plot_level=None):
    # Create file name     
    sd_i = 'Global' if varname == 'MOC' else sd_str
    domain_str = sd_i
    if r_domain:
        domain_str += '.'+rd_str
    if o_domain:
        domain_str += '.'+od_str
    filename = plot_type+'.'+varname+'.'+domain_str+'.'+td_str+'.'+tavg+'.'+ens_str+'.nc'
    filename2 = plot_type+'.'+varname+'.'+domain_str+'.'+td_str+'.'+tavg+'.All_members.nc'
    filename3 = plot_type+'.'+varname+'.'+sd_i+'.'+'1950'+'.'+tavg+'.'+ens_str+'.nc'
    filepath = path_to_data+filename

    # If file exists, open
    if Path(filepath).exists():
        print(filename)
        data = xr.open_dataset(filepath)

    # If ensemble mean and version with all members exists, open and take ensemble mean
    elif ens_type and Path(path_to_data+filename2).exists():
        print(filename2)
        data = xr.open_dataset(path_to_data+filename2)
        data = data.mean('ensemble_member')

    # If ensemble mean and version starting in 1950 exists, open
    elif ens_type and Path(path_to_data+filename3).exists():
        print(filename3)
        data = xr.open_dataset(path_to_data+filename3)
    
    # Else, requested ensemble or time domain type doesn't exist but version with all members and starting in 1950 does
    else:
        filename = plot_type+'.'+varname+'.'+sd_i+'.'+'1950'+'.'+tavg+'.All_members.nc'
        filepath = path_to_data+filename
        print(filename)
        data = xr.open_dataset(filepath)
        if ens_type:
            data = data.mean('ensemble_member')
    
    return data

#### SubCmap

In [11]:
def SubCmap(orgcmap, levels, type, color):
    # Type is middle, end or beginning
    # Make cmap values swapped out
    # color is color to sub in
    lev_len = len(levels)+1
    cmap_samp = orgcmap.resampled(lev_len)
    cmap_list = cmap_samp(np.linspace(0,1,lev_len))

    # Swap out first two
    if type == 'beg':
        lev_ind = 0
        cmap_list[lev_ind:(lev_ind+1), :] = color[0]
        cmap_list[(lev_ind+1):(lev_ind+2), :] = color[1]
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [color[0], orgcmap.get_over()], True)

    # Swap out middle two
    elif type == 'mid':
        lev_ind = int(lev_len/2)
        cmap_list[(lev_ind-1):(lev_ind+1), :] = color
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [orgcmap.get_under(), orgcmap.get_over()], True)

    # Swap out last
    else:
        lev_ind = -1
        cmap_list[lev_ind:, :] = color
        final_cmap, _ = CustomCmap(levels, cmap_list, 
                           [orgcmap.get_under(), color], True)

    return final_cmap

#### TrendLine

In [12]:
def TrendLine(da, period):
    da = da.dropna(dim=period)
    # Calculate regression coefficients
    da_reg_coef = da.polyfit(dim=period, deg=1)

    # Calculate trend line
    da_yreg = (da_reg_coef.loc[dict(degree=1)]*da[period])+da_reg_coef.loc[dict(degree=0)]

    return da_yreg['polyfit_coefficients']

#### DetrendAMOC

In [13]:
def DetrendAMOC(ds, ds2, ty, v):
    # ds: aice or TREFHT, ds2: AMOC, ty: var 1 annual (1) or monthly (0) variable
    print('Detrending '+v+' with '+var2)
    control_name = 'LENS2 piControl'
    if ty == 1:
        # Remove nans
        ds2_nan = ds2[control_name].values.flatten()
        ds2_nan = ds2_nan[~np.isnan(ds2_nan)]

        ds_nan = ds[control_name].values.flatten()
        ds_nan = ds_nan[~np.isnan(ds_nan)]
        
        # Create linear regression
        m, b, r, p, se = stats.linregress(ds2_nan, ds_nan)
        print('   slope: {:.3f}'.format(m))
        print('   inter: {:.3f}'.format(b))
        print('   r-val: {:.3f}'.format(r))
        print('   p-val: {:.3f}'.format(p))

        # Calculate effect of AMOC in sea ice and temperature
        ds_amoc_eff = m*ds2
        if v == 'aice':
            ds_amoc_eff = ds_amoc_eff.rename({'year':'time'})
            ds_amoc_eff = ds_amoc_eff.assign_coords({'time':ds.time})

        # Remove effect of AMOC
        ds_wo_amoc = ds-ds_amoc_eff

        # Add observed variables back in
        for dsname, attrs in ds_names.items():
            if attrs[1] == 3 and attrs[0] and dsname in ds:
                ds_wo_amoc[dsname] = ds[dsname]
    else:
        ds_list = []
        ds2_list = []
        # Remove nans
        ds2_nan = ds2[control_name].values.flatten()
        ds2_nan = ds2_nan[~np.isnan(ds2_nan)]
        for m in np.arange(1,13):
            ds_m = ds.loc[{'month':m}]
            print(month_str[m-1])

            # Remove nans
            ds_m_nan = ds_m[control_name].values.flatten()
            ds_m_nan = ds_m_nan[~np.isnan(ds_m_nan)]
            
            # Create linear regression
            m, b, r, p, se = stats.linregress(ds2_nan, ds_m_nan)
            print('   slope: {:.3f}'.format(m))
            print('   inter: {:.3f}'.format(b))
            print('   r-val: {:.3f}'.format(r))
            print('   p-val: {:.3f}'.format(p))
    
            # Calculate effect of AMOC in sea ice and temperature
            ds_m_amoc_eff = m*ds2
    
            # Remove effect of AMOC
            ds_m_wo_amoc = ds_m-ds_m_amoc_eff

            # Add observed variables back in
            for dsname, attrs in ds_names.items():
                if attrs[1] == 3 and attrs[0] and dsname in ds:
                    ds_m_wo_amoc[dsname] = ds_m[dsname]
                    
            ds_list.append(ds_m_wo_amoc)
            ds2_list.append(ds_m_amoc_eff)

        ds_wo_amoc = xr.concat(ds_list, dim=pd.Index(np.arange(1,13), name='month'))
        ds_amoc_eff = xr.concat(ds2_list, dim=pd.Index(np.arange(1,13), name='month'))
    
    return ds_wo_amoc, ds_amoc_eff

#### AttrCalc

In [14]:
def AttrCalc(ds, varname, skip_obs):
    use_obs = varname != 'MOC'  and varname != 'AEROD_v' and not skip_obs
    if use_obs:
        obs_name_list = [name for name, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]

    funcs = {0: np.subtract,
            1: np.add}

    # 0: subtract, 1: addition
    attr_names = {'Non-GHG': ['HIST_UVnudge_LM',0,'GHG_UVnudge_LM'],
                 'Anthropogenic forcing': ['HIST_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Greenhouse gases': ['GHG_UVnudge_LM',0,'PiC_UVnudge_LM2006'],
                 'Winds': ['PiC_UVnudge_LM2006'],
                 'HIST+winds': ['HIST_UVnudge_LM'],
                 'HIST-control': ['LENS2 HIST-SSP370'],
                 'Greenhouse gases-control': ['GHG2'],
                 'Non-GHG-control': ['LENS2 HIST-SSP370',0,'GHG2']}
    if use_obs:
        for obs_name in obs_name_list:
            attr_names[ds_paper_names[obs_name]] = [obs_name]
            attr_names['Bias ('+ds_paper_names[obs_name][5:-1]+')'] = [obs_name,0,'HIST_UVnudge_LM']
    
    attr_list = []
    
    for aname, eq in attr_names.items():
        print(aname)
        # If term composed of one variable - no calculation needed
        if len(eq) == 1 and ds_names[eq[0]][0]:
            eqname = eq[0]+' mean' if ds_names[eq[0]][1] == 0 else eq[0]
            attr_list.append(ds[eqname].rename(aname))
            print('  Calculated')

        # If calculation needs two terms
        elif len(eq) > 1:
            # Find all variable names in calculation
            calc_names = eq[::2]

            # See if all names in calculation are set to be plotted, if not don't calculate this attribution term
            all = True
            for cname in calc_names:
                if not ds_names[cname][0]:
                    all = False
                    break
            if not all:
                continue
            else:
                eqname = eq[0] + ' mean' if ds_names[eq[0]][1] == 0 else eq[0]
                da_attr = ds[eqname]
                for i in np.arange(1,len(eq)-1,2):
                    eqname = eq[i+1] + ' mean' if ds_names[eq[i+1]][1] == 0 else eq[i+1]
                    da_attr = xr.apply_ufunc(funcs[eq[i]],da_attr,ds[eqname])
                attr_list.append(da_attr.rename(aname))
                print('  Calculated')

    ds_attr = xr.merge(attr_list)
    return ds_attr

#### CalcR

In [15]:
def CalcR(ds_attr):
    da_hist = ds_attr['HIST+winds']
    da_hist_short = da_hist.loc[dict(start_year=slice(1980,2005))]
    corr_dict = {}

    for aname, da in ds_attr.items():
        if 'OBS' in aname:
            if 'NSIDC' in aname or 'OSISAF' in aname:
                corr = xr.corr(da_hist_short, da.loc[dict(start_year=slice(1980,2005))])
            else:
                corr = xr.corr(da_hist, da)
            corr_dict[aname] = corr.values
    return corr_dict  

#### AddMidYear

In [16]:
def AddMidYear(ds_trd):
    ds_trd = ds_trd.assign_coords(mid_year=('start_year',ds_trd.start_year.values+10))
    return ds_trd

#### CalcPIStatSig

In [17]:
def CalcPIStatSig(da_pi, ds_attr):
    # Calculate pre-industrial statistics
    pimean = da_pi.mean('slice', skipna=True)
    pistd = da_pi.std('slice', skipna=True)

    # Z statistic
    zstat = (ds_attr-pimean)/pistd

    # P-value
    dz_list = []
    for daname, dz in zstat.items():
        pval = stats.norm.sf(abs(dz.values))**2
        dpval = dz.copy(data=pval)
        dz_list.append(dpval)
    
    ds_pval = xr.merge(dz_list)
    return ds_pval

#### LoadRegridder

In [18]:
def LoadRegridder():
    filestem = 'ERAregridder'
    method = 'bilinear'
    
    # Check for existing target grid
    targetgridpath = path_to_data+filestem+'.'+sd_str+'.targetgrid.nc'

    # Check for existing sample grid
    samplegridpath = path_to_data+filestem+'.'+sd_str+'.samplegrid.nc'

    # Check for existing regridder weights
    weightpath = path_to_data+filestem+'.'+sd_str+'.weights.nc'

    # If exists, open file for target grid
    target_grid = xr.open_dataset(targetgridpath)

    # If exists, open file
    sample_grid = xr.open_dataset(samplegridpath)
   
    regridder = xe.Regridder(sample_grid, target_grid, method, filename=weightpath,reuse_weights=True)

    return regridder

#### RegridERA5

In [19]:
def RegridERA5(ds, regridder):
    ## Regridding
    print('Regridding ERA5 grid -> ATM grid...')
    da = ds.rename({'latE': 'lat', 'lonE': 'lon'})
    da = da.where(da.lon != 180.0, drop=True)
    da_re = regridder(da)
    da_re = da_re.rename({'x':'lon', 'y':'lat'})
    da_re = da_re.assign_coords({'lon':lons, 'lat': arclats})
    da_re = da_re.rename('ERA5')
    

    # Adding back in cyclic point
    cyclic_data, cyclic_lon = add_cyclic_point(da_re.data, coord=da_re['lon'])
    cyclic_coords = {dim: da_re.coords[dim] for dim in da_re.dims}
    cyclic_coords['lon'] = cyclic_lon
    da_re = xr.DataArray(cyclic_data, dims=da_re.dims, coords=cyclic_coords, attrs=da_re.attrs, name=da_re.name)
    
    return da_re

#### FigLabelList

In [20]:
def FigLabelList(total):
    # total = total number of letters needed for subfigure label list
    lowercase = string.ascii_lowercase
    label_list = ['('+lowercase[i]+')' for i in range(0,total)]
    return label_list

#### AxisLabels

In [21]:
def AxisLabels(ax, title_str, land):
    if land:
        ax.add_feature(cfeature.LAND, zorder=3)
    ax.coastlines(zorder=4)
    ax.set_extent(extent, ccrs.PlateCarree())
    ax.set_title(title_str, fontsize = 10)
    draw_circle(ax, draw_circ=(s_domain == 1), draw_major=False)
    return None

#### SaveFig

In [22]:
def SaveFig(fig, plot_type, tavg, varname, note=None, plot_level=None):
    # File format is:
    # var.ensemble_type.spatialdomain.[Zplot_level].plot_type.timeaveraging.timedomain.[note].png
    level_str = '' if plot_level == None else 'Z'+str(plot_level)+'.'
    note_str = '' if note == None else note+'.'
    
    fig.savefig(path_to_graphs+varname+'.'+ens_str+'.'+sd_str+'.'+level_str+plot_type+'.'+tavg+'.'+td_str+'.'+note_str+'pdf', bbox_inches='tight')
    return None

## Month trend plots
### Set up

In [ ]:
if plot_types['mtrd']:
    graph_type_str = 'Linear.Trend' 

    # Graph labels and variables
    date_str = 'Month'
    period = 'month'
    dim_avg = 'time.month'
        
    trdyr_str = '.'+str(trd_length)+'yr'
    ramoc_str = '.DtrdAMOC' if remove_amoc else ''
    if o_domain:
        ramoc_str += '.'+str(ocn_lat)+'N'
    else:
        ramoc_str += '.33-55N'
    
    mcmaps = {'aice': cm.vik_r,
                'TREFHT': cm.vik,
                'MOC': cm.vik}
    mdcmaps = {'aice': cm.cork,
                'TREFHT': cm.cork}
    mlevels = {'aice': np.arange(-1.80,1.81,0.15),
                'TREFHT': np.arange(-3,3.1,0.25),
                'MOC': np.arange(-4,4.01,0.25)}
    mticks = {'aice': np.arange(-1.8,1.81,0.6),
                'TREFHT': np.arange(-3,3.1,1.0),
                'MOC': np.arange(-4,4.01,1.0)}
    mlabels = {'aice': np.arange(-1.5,1.6,0.5),
                'TREFHT': np.arange(-2,2.1,1.0)}
    

    xlim = [t_domain+10,t_end-(trd_length-1)+10]
    ylim = {'aice': [-2,1],
            'TREFHT': [-2,2],
            'MOC': [-4,4]}
    ylimcontr = {'aice': [-2.2,1.2],
                'TREFHT': [-1.2,1.8],
                'MOC': [-5,5]}
    ywidth = ((t_end-(trd_length-1))-t_domain)*(5/55)
    ylabelsingle = {'aice': mon_str[type_onemonth-1]+' sea ice extent trend (million km$^2$/decade)',
                    'TREFHT': 'Annual 2m air temperature trend (K/decade)',
                    'MOC': 'Annual Atlantic Meridional Overturning Circulation trend (Sv/decade)'}
    ylabel = {'aice': 'Sea ice extent trend (million km$^2$/decade)',
                'TREFHT': '2m air temperature trend (K/decade)'}
    ylabel2line = {'aice': 'Sea ice extent trend\n(million km$^2$/decade)',
                    'TREFHT': '2m air temperature\ntrend (K/decade)'}
    
    xlabel = 'Mid year of '+str(trd_length)+'-year trend'
    xdim = 'mid_year'
    bbox_loc = {'aice': 'lower left',
                'TREFHT': 'lower right',
                'MOC': 'lower left'}

### Data loading

In [ ]:
%%time

if plot_types['mtrd']:
    
    if var != 'MOC':
        # Load monthly trends
        ds_mon_trd = LoadData(graph_type_str+'.'+trdyr_str, var, 'month')
        ds_mon_trd = AddMidYear(ds_mon_trd)

    # Load trends
    ds_ann_trd = LoadData(graph_type_str+'.'+trdyr_str, var, 'year')
    ds_ann_trd = AddMidYear(ds_ann_trd)

    if var != 'MOC':
        obs_name_list = [ds_paper_names[name] for name, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]
        obs_len = len(obs_name_list)
        for i in range(0,obs_len*2,2):
            obs_name = obs_name_list[int(i/2)]
            ds_attr_names[obs_name] = [i+2, 'k']
            ds_attr_names['Bias ('+obs_name[5:-1]+')'] = [i+3, 'orchid']
            
        if var != 'aice':
            ds_attr_ann = AttrCalc(ds_ann_trd, var, False)
            ds_pval_ann = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann)
        ds_attr_mon = AttrCalc(ds_mon_trd, var, False)
        ds_pval_mon = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_attr_mon)
    else:
        ds_attr_ann = AttrCalc(ds_ann_trd, var, False)
        ds_pval_ann = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann)

CPU times: user 4 µs, sys: 1 µs, total: 5 µs
Wall time: 6.91 µs


In [ ]:
%%time

# Secondary variable
if plot_types['mtrd']:
    # Load secondary variable - MOC
    # Load annual trends
    ds2_ann_trd = LoadData(graph_type_str+'.'+trdyr_str, var2, 'year')   
    ds2_ann_trd = AddMidYear(ds2_ann_trd)
    ds2_ann_trd = ds2_ann_trd.sel(lat=ocn_lat, method='nearest')

    ds2_attr_ann = AttrCalc(ds2_ann_trd, var2, False)
    ds2_pval_ann = CalcPIStatSig(ds2_ann_trd['LENS2 piControl'], ds2_attr_ann)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.96 µs


In [ ]:
%%time

# Remove effect of AMOC
if plot_types['mtrd'] and remove_amoc:
    # Remove AMOC 
    if var != 'aice':
        ds_ann_trd_amoc, ds_ann_trd_amoc_eff = DetrendAMOC(ds_ann_trd, ds2_ann_trd, 1, var)
        ds_attr_ann_amoc = AttrCalc(ds_ann_trd_amoc, var, False)
        ds_pval_ann_amoc = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_attr_ann_amoc)
        ds_pval_ann_amoc_eff = CalcPIStatSig(ds_ann_trd['LENS2 piControl'], ds_ann_trd_amoc_eff)
    
    ds_mon_trd_amoc, ds_mon_trd_amoc_eff = DetrendAMOC(ds_mon_trd, ds2_ann_trd, 0, var)
    ds_attr_mon_amoc = AttrCalc(ds_mon_trd_amoc, var, False)
    ds_pval_mon_amoc = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_attr_mon_amoc)
    ds_pval_mon_amoc_eff = CalcPIStatSig(ds_mon_trd['LENS2 piControl'], ds_mon_trd_amoc_eff)

CPU times: user 3 µs, sys: 0 ns, total: 3 µs
Wall time: 5.72 µs


In [27]:
if plot_types['mtrd']  and var != 'MOC':
    patt_corr_mon = CalcR(ds_attr_mon)
    
    print('Pattern correlations (with AMOC)')
    for name, corr in patt_corr_mon.items():
        print('  '+name+': '+format(corr, '.2f'))

    if remove_amoc:
        patt_corr_mon_amoc = CalcR(ds_attr_mon_amoc)
    
        print('Pattern correlations (without AMOC)')
        for name, corr in patt_corr_mon_amoc.items():
            print('  '+name+': '+format(corr, '.2f'))

In [28]:
# Set variables according to AMOC removal status
if plot_types['mtrd'] and var != 'MOC':
    if remove_amoc:
        if var != 'aice':
            dsM_ann_trd = ds_ann_trd_amoc
            dsM_attr_ann = ds_attr_ann_amoc
            dsM_pval_ann = ds_pval_ann_amoc
        dsM_mon_trd = ds_mon_trd_amoc
        dsM_attr_mon = ds_attr_mon_amoc
        dsM_pval_mon = ds_pval_mon_amoc
        Mpatt_corr_mon = patt_corr_mon_amoc
        
    elif not remove_amoc:
        if var != 'aice':
            dsM_ann_trd = ds_ann_trd
            dsM_attr_ann = ds_attr_ann
            dsM_pval_ann = ds_pval_ann
        dsM_mon_trd = ds_mon_trd
        dsM_attr_mon = ds_attr_mon
        dsM_pval_mon = ds_pval_mon
        Mpatt_corr_mon = patt_corr_mon

### Make plots

#### Running trends plots
##### Attribution Contour plots

In [ ]:
%%time

if plot_types['mtrd'] and ens_type and comp != 'ocn':
    # Framework
    frame = {'model': ['HIST+winds','Greenhouse gases','Non-GHG','Winds']}
    if var != 'MOC'  and var != 'AEROD_v':
        frame['model+obs'] = ['HIST+winds']
        for obs_name in obs_name_list:
            frame['model+obs'].append(obs_name)
            frame['model+obs'].append('Bias ('+obs_name[5:-1]+')')
    if r_domain:
        regions = ds_attr_mon.region.values
    else:
        regions = np.array(['A'])
        
    # Setup
    colPlot = 2

    for r in regions:
        rdict = dict(region=r) if r_domain else dict()
        dr_attr_mon = dsM_attr_mon.loc[rdict]
        dr_pval_mon = dsM_pval_mon.loc[rdict]
        addon = ''
    
        for f, fvlist in frame.items():
            df_attr_mon = dr_attr_mon[fvlist]
            df_pval_mon = dr_pval_mon[fvlist]
            
            numVar = len(df_attr_mon.data_vars)
            chgt = int(np.ceil(numVar/colPlot))
            totalPlot = colPlot*chgt
            hr = np.ones(chgt+1)
            hr[-1] = 0.1
            let_labels = FigLabelList(totalPlot)
            
            fig, axlist = plt.subplots(chgt+1, colPlot, height_ratios=hr, layout='constrained')
            fig.set_size_inches(ywidth*colPlot,3.5*chgt)
            axlist = axlist if chgt == 1 else list(itertools.chain.from_iterable(axlist))
            gs = axlist[-2].get_gridspec()
            for ax in axlist[-2:]:
                ax.remove()
            cbar_ax = fig.add_subplot(gs[-2:])
        
            for daname, da_attr in df_attr_mon.items():
                p = ds_attr_names[daname][0]
                da_pval = df_pval_mon[daname]

                patt_add = ','+str(round(float(Mpatt_corr_mon[daname]),2)) if 'OBS' in daname else ''
            
                # Plot data
                if 'NSIDC' not in daname and 'OSISAF' not in daname:
                    cax = da_attr.plot.contourf(
                            ax=axlist[p], x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                            add_colorbar=False, zorder=1)
                    # clab = da_attr.plot.contour(
                    #         ax=axlist[p], x=xdim, y=period, levels=mlabels[var],
                    #         colors='k', zorder=2)
                    da_pval.plot.contourf(
                        ax=axlist[p], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                        add_colorbar=False, zorder=2)
                    axlist[p].set_xlabel(xlabel, fontsize=14)
                    axlist[p].set_ylabel(date_str, fontsize=14)
                    axlist[p].set_yticks(ticks=np.arange(1,13), labels=m_str)
                    axlist[p].set_xlim(xlim)
                    axlist[p].set_title(daname+patt_add, fontsize=15)
                    axlist[p].tick_params(labelsize=12)
                    # axlist[p].clabel(clab, clab.levels, fontsize=12)
                    axlist[p].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')

                else:
                    axin = inset_axes(axlist[p], width='46.4%', height='100%', loc='center right', borderpad=0)
                    cax = da_attr.plot.contourf(
                            ax=axin, x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                            add_colorbar=False, zorder=1)
                    # clab = da_attr.plot.contour(
                    #         ax=axin, x=xdim, y=period, levels=mlabels[var],
                    #         colors='k', zorder=2)
                    da_pval.plot.contourf(
                            ax=axin, x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                            add_colorbar=False, zorder=2)
                    axin.set_xlabel(xlabel, fontsize=14)
                    axin.set_ylabel(date_str, fontsize=14)
                    axin.set_yticks(ticks=np.arange(1,13), labels=m_str)
                    axin.tick_params(labelsize=12)
                    # axin.clabel(clab, clab.levels, fontsize=12)
                    axin.set_xlim([1990,t_end-(trd_length-1)+10])
                    axin.set_title(daname+patt_add, fontsize=15)
                    axin.annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')
                    axlist[p].set_ylabel(' ', fontsize=14)
                    axlist[p].set_xlabel(' ', fontsize=14)
                    axlist[p].set_title(' ', fontsize=18)
                    axlist[p].spines['top'].set_visible(False)
                    axlist[p].spines['right'].set_visible(False)
                    axlist[p].spines['bottom'].set_visible(False)
                    axlist[p].spines['left'].set_visible(False)
                    axlist[p].tick_params(labelsize=12, length=0, width=0)
                    axlist[p].set_xticks(np.arange(0,1.1,1), labels=[' ',' '])
                    axlist[p].set_yticks(np.arange(0,1.1,1), labels=[' ',' '])
                    
                
        
            if numVar < totalPlot:
                axlist[1].axis('off')
        
            # Formatting
            cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='bottom')
            cb.set_label(label=ylabel[var], fontsize=14)
            cb.ax.tick_params(labelsize=12)
            cb.set_ticks(ticks=mticks[var])

            if r_domain:
                fig.suptitle(region_names[r],fontsize=16)
        
            SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-month-attribute.'+f+addon, var, note)
        
            plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.44 µs


##### AMOC Effect

In [ ]:
%%time

# Load other variable
if plot_types['mtrd'] and ens_type and remove_amoc:
    var3 = 'aice' if var == 'TREFHT' else 'TREFHT'
    ds3_mon_trd = LoadData(graph_type_str+'.'+trdyr_str, var3, 'month')
    ds3_mon_trd = ds3_mon_trd.assign_coords(mid_year=('start_year',ds3_mon_trd.start_year.values+10))
    ds3_attr_mon = AttrCalc(ds3_mon_trd, var3, True)
    ds3_pval_mon = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_attr_mon)
    
    ds3_mon_trd_amoc, ds3_mon_trd_amoc_eff = DetrendAMOC(ds3_mon_trd, ds2_ann_trd, 0, var3)
    ds3_attr_mon_amoc = AttrCalc(ds3_mon_trd_amoc, var3, True)
    ds3_pval_mon_amoc = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_attr_mon_amoc)
    ds3_pval_mon_amoc_eff = CalcPIStatSig(ds3_mon_trd['LENS2 piControl'], ds3_mon_trd_amoc_eff)

CPU times: user 3 µs, sys: 1 µs, total: 4 µs
Wall time: 5.72 µs


In [ ]:
%%time

if plot_types['mtrd'] and ens_type and remove_amoc:
    # Framework    
    frame = {'HIST+winds (with AMOC)': [0, 'HIST+winds',[ds_attr_mon,ds_pval_mon,
                                                         ds3_attr_mon,ds3_pval_mon]],
             'HIST+winds (without AMOC)': [1, 'HIST+winds',[ds_attr_mon_amoc,ds_pval_mon_amoc,
                                                            ds3_attr_mon_amoc,ds3_pval_mon_amoc]],
             'AMOC Effect': [2, 'HIST_UVnudge_LM',[ds_mon_trd_amoc_eff,ds_pval_mon_amoc_eff,
                                                   ds3_mon_trd_amoc_eff,ds3_pval_mon_amoc_eff]]}
    # Setup
    colPlot = 3
    numVar = 6
    chgt = int(np.ceil(numVar/colPlot))
    totalPlot = colPlot*chgt
    wr = np.ones(colPlot+1)
    wr[-1] = 0.05
    let_labels = FigLabelList(totalPlot)
    
    fig, axlist = plt.subplots(chgt, colPlot+1, width_ratios=wr, layout='constrained')
    fig.set_size_inches((ywidth*colPlot),3.5*chgt)
    gs = axlist[0,-1].get_gridspec()
    for ax in axlist[:,-1]:
        ax.remove()
    cbar_ax = fig.add_subplot(gs[0,-1])
    cbar_ax3 = fig.add_subplot(gs[1,-1])

    for label, flist in frame.items():
        c = flist[0]
        dname = flist[1]
        da = flist[2][0][dname]
        da_pval = flist[2][1][dname]
        da3 = flist[2][2][dname]
        da3_pval = flist[2][3][dname]
    
        # Plot data
        cax = da.plot.contourf(
                ax=axlist[0,c], x=xdim, y=period, levels=mlevels[var], cmap=mcmaps[var],
                add_colorbar=False, zorder=1)
        da_pval.plot.contourf(
                ax=axlist[0,c], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)

        cax3 = da3.plot.contourf(
                ax=axlist[1,c], x=xdim, y=period, levels=mlevels[var3], cmap=mcmaps[var3],
                add_colorbar=False, zorder=1)
        da3_pval.plot.contourf(
                ax=axlist[1,c], x=xdim, y=period, levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)
        
        for r in range(0,2):
            axlist[r,c].set_xlabel(xlabel, fontsize=14)
            axlist[r,c].set_ylabel(date_str, fontsize=14)
            axlist[r,c].set_yticks(ticks=np.arange(1,13), labels=m_str)
            axlist[r,c].set_xlim(xlim)
            axlist[r,c].set_title(label, fontsize=15)
            axlist[r,c].tick_params(labelsize=12)
            p = r*colPlot+c
            axlist[r,c].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')

    if numVar < totalPlot:
        for p in range(numVar,totalPlot):
            axlist[p].axis('off')

    # Formatting
    cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='right')
    cb.set_label(label=ylabel2line[var], fontsize=14)
    cb.ax.tick_params(labelsize=12)
    cb.set_ticks(ticks=mticks[var])

    cb3 = fig.colorbar(cax3, cax=cbar_ax3, shrink=0.1, extend='both', location='right')
    cb3.set_label(label=ylabel2line[var3], fontsize=14)
    cb3.ax.tick_params(labelsize=12)
    cb3.set_ticks(ticks=mticks[var3])

    SaveFig(fig, graph_type_str+ramoc_str, trdyr_str+'-trend-month-amoc', var+'-'+var3, note)

    plt.close(fig)

CPU times: user 4 µs, sys: 1e+03 ns, total: 5 µs
Wall time: 13.8 µs


##### AMOC trends by latitude

In [ ]:
%%time

if plot_types['mtrd'] and ens_type and comp == 'ocn' and o_domain:
    # Framework
    frame = {'model': ['HIST+winds','Greenhouse gases','Non-GHG','Winds']}
        
    # Setup
    colPlot = 2
    
    for f, fvlist in frame.items():
        df_attr_ann = ds_attr_ann[fvlist]
        df_pval_ann = ds_pval_ann[fvlist]
        
        numVar = len(df_attr_ann.data_vars)
        chgt = int(np.ceil(numVar/colPlot))
        totalPlot = colPlot*chgt
        hr = np.ones(chgt+1)
        hr[-1] = 0.1
        let_labels = FigLabelList(totalPlot)
        
        fig, axlist = plt.subplots(chgt+1, colPlot, height_ratios=hr, layout='constrained')
        fig.set_size_inches(ywidth*colPlot,3.5*chgt)
        axlist = axlist if chgt == 1 else list(itertools.chain.from_iterable(axlist))
        gs = axlist[-2].get_gridspec()
        for ax in axlist[-2:]:
            ax.remove()
        cbar_ax = fig.add_subplot(gs[-2:])
    
        for daname, da_attr in df_attr_ann.items():
            p = ds_attr_names[daname][0]
            da_pval = df_pval_ann[daname]
        
            # Plot data
            cax = da_attr.plot.contourf(
                    ax=axlist[p], x=xdim, y='lat', levels=mlevels[var], cmap=mcmaps[var],
                    add_colorbar=False, zorder=1)
            da_pval.plot.contourf(
                ax=axlist[p], x=xdim, y='lat', levels=[-0.01,0.05,1.0],hatches=[None,'..'],colors='none',
                add_colorbar=False, zorder=2)
            axlist[p].set_xlabel(xlabel, fontsize=14)
            axlist[p].set_ylabel('Latitude', fontsize=14)
            axlist[p].set_yticks(ticks=np.arange(20,61,10))
            axlist[p].set_xlim(xlim)
            axlist[p].set_title(daname, fontsize=15)
            axlist[p].tick_params(labelsize=12)
            # axlist[p].clabel(clab, clab.levels, fontsize=12)
            axlist[p].annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontsize=15, fontweight='bold')           
            
    
        if numVar < totalPlot:
            axlist[1].axis('off')
    
        # Formatting
        cb = fig.colorbar(cax, cax=cbar_ax, shrink=0.1, extend='both', location='bottom')
        cb.set_label(label=ylabelsingle[var], fontsize=14)
        cb.ax.tick_params(labelsize=12)
        cb.set_ticks(ticks=mticks[var])
    
        SaveFig(fig, graph_type_str, trdyr_str+'-trend-month-attribute.'+f, var, note)
    
        plt.close(fig)

CPU times: user 4 µs, sys: 0 ns, total: 4 µs
Wall time: 6.91 µs


## Line and running trends plots
### Set up

In [ ]:
if plots['comb'][0]:
    graph_type_str = 'Linear.Trend.abs-20yr-40yr'
    hgt = 3
    wdth = 3

    if var == 'MOC' and o_domain and not ens_type:
        graph_type_str += '.'+str(ocn_lat)+'N'
    elif var == 'MOC' and o_domain and ens_type:
        graph_type_str += '.20-55N'

    ylabelabs = {'TREFHT': 'Annual 2m air\ntemperature (K)',
                 'MOC': 'AMOC (Sv)',}
    ylabelabs1m = {'aice': ' sea\nice extent (million km$^2$)',}
    ylimabs = {'aice': {3: [13,17],
                        9: [0,10]},
               'TREFHT': [254,265],
               'MOC': [10,28]}
    yticksabs = {'aice': {3: np.arange(13,18,1),
                          9: np.arange(0,11,2)},
                'TREFHT': np.arange(254,266,3),
                'MOC': np.arange(10,28,5)}
    
    # Prepare x ticks
    # Timeseries
    if time_avg == 4:
        period ='time'
        date_str = mon_str[type_onemonth-1]
        
        xlimabs = [t_domain,2025]
        xlabelabs = 'Year' 
        ylabelabs[var] = month_str[type_onemonth-1]+ylabelabs1m[var]
        if var == 'aice':
            ylimabs['aice'] = ylimabs['aice'][type_onemonth]
            yticksabs['aice'] = yticksabs['aice'][type_onemonth]
        
    # Yearly
    elif time_avg == 1:
        period ='year'
        date_str = 'year'
        
        xlimabs = [t_domain,2024]
        xlabelabs= 'Year'

    xticks_loc = np.arange(1950,2021,10)
    xticks_locmin = np.arange(1950,2025,5)
    xlimtrd = lambda tl: [t_domain+10,t_end-(tl-1)+10]
    ylimtrd20 = {'aice': [-2,1],
           'TREFHT': [-2,2],
           'MOC': [-4,4],}
    ylimtrd40 = {'aice': [-1.2,0.5],
           'TREFHT': [-0.5,1.2],
           'MOC': [-3,3],}
    ywidth = lambda tl: ((t_end-(tl-1))-t_domain)*(3/55) # 20, 40, 1 (abs)
    ylabeltwolinesingle = {'aice': mon_str[type_onemonth-1]+' sea ice extent\ntrend (million km$^2$/decade)',
                    'TREFHT': 'Annual 2m air temperature\ntrend (K/decade)',
                   'MOC': 'AMOC (Sv/decade)',}
    xlabeltrd = lambda tl: 'Mid year of '+str(tl)+'-year trend'
    xdim = 'mid_year'

### Data loading

In [51]:
%%time

if plots['comb'][0]:
    ## Absolute (only one that will be used for SIA-one month, TOA, AMOC, drift)
    # Yearly
    if time_avg == 1:
        ds_abs = LoadData('Linear.abs', var, period)

    # Timeseries of one month of sea ice area data
    elif time_avg == 4:
        # If only plotting one month
        ds_abs = LoadData('Linear.abs', var, mon_str[type_onemonth-1])

    ## Load trend data
    # If var is sea ice
    if var == 'aice':
        # Load 20-year trends
        ds_mon_trd = LoadData('Linear.Trend.20yr', var, 'month')
        ds_20trd = ds_mon_trd.loc[dict(month=type_onemonth)]

        # Load 40-year trends
        ds_mon_trd = LoadData('Linear.Trend.40yr', var, 'month')
        ds_40trd = ds_mon_trd.loc[dict(month=type_onemonth)]

    # Else var is temperature or AMOC
    else:
        # Load 20-year trends
        ds_20trd = LoadData('Linear.Trend.20yr', var, 'year')

        # Load 40-year trends
        ds_40trd = LoadData('Linear.Trend.40yr', var, 'year')

    ds_20trd = AddMidYear(ds_20trd)
    ds_40trd = AddMidYear(ds_40trd)

    if var == 'MOC' and o_domain and not ens_type:
        ds_abs = ds_abs.sel(lat=ocn_lat, method='nearest')
        ds_20trd = ds_20trd.sel(lat=ocn_lat, method='nearest')
        ds_40trd = ds_40trd.sel(lat=ocn_lat, method='nearest')
    elif var == 'MOC' and o_domain and ens_type:
        ds_abs_def = xr.open_dataset(path_to_data+'Linear.abs.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')
        ds_20trd_def = xr.open_dataset(path_to_data+'Linear.Trend.20yr.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')
        ds_40trd_def = xr.open_dataset(path_to_data+'Linear.Trend.40yr.MOC.Global.1950.year.All_members.nc').mean('ensemble_member')

        ds_20trd_def = AddMidYear(ds_20trd_def)
        ds_40trd_def = AddMidYear(ds_40trd_def)

Linear.abs.aice.Arctic.1950.Sep.All_members.nc
Linear.Trend.20yr.aice.Arctic.1950.month.All_members.nc
Linear.Trend.40yr.aice.Arctic.1950.month.All_members.nc
CPU times: user 34.3 ms, sys: 3.8 ms, total: 38.1 ms
Wall time: 85.5 ms


### Make plots

In [52]:
%%time

if plots['comb'][0]:
    obs_list = [dsname for dsname, attrs in ds_names.items() if (attrs[0] and attrs[1] == 3)]
    
    row_dict = {0: ['LENS2 piControl','PiC_UVnudge_LM2006']+obs_list,
                1: ['GHG2', 'GHG_UVnudge_LM']+obs_list,
                2: ['LENS2 HIST-SSP370', 'HIST_UVnudge_LM']+obs_list}
                # List: [0:dataset, 1:x dimension, 2:trend length, 3:y-label, 4:y-limit, 5:x-label, 6:x-limit]
    col_dict = {0: [ds_abs, period, 1, ylabelabs[var], ylimabs[var], xlabelabs, xlimabs],
                1: [ds_20trd, xdim, 20, ylabeltwolinesingle[var], ylimtrd20[var], xlabeltrd(20), xlimtrd(20)],
                2: [ds_40trd, xdim, 40, ylabeltwolinesingle[var], ylimtrd40[var], xlabeltrd(40), xlimtrd(40)]}
    
    # Set up figure and subplots
    fig = plt.figure(figsize=(11,8),layout='constrained')
    gs = gridspec.GridSpec(hgt+1, wdth, height_ratios=[1,1,1,0.3],
                            width_ratios=[ywidth(1),ywidth(20),ywidth(40)],
                            wspace=0.4, hspace=0.6)
    hand_list = []
    label_list = []
    let_labels = FigLabelList(hgt*wdth)
    
    # Plot all data
    for r, dnames in row_dict.items():
        # Create axes for given row
        axa = fig.add_subplot(gs[r,0])
        ax20 = fig.add_subplot(gs[r,1])
        ax40 = fig.add_subplot(gs[r,2])
        axlist = [axa, ax20, ax40]

        for c, clist in col_dict.items():
            ax = axlist[c]
            ds = clist[0]
            if c != 0:
                ax.axhline(0, color='k', zorder=0, lw=0.5)
                
            for dsname in dnames:
                attrs = ds_names[dsname]
                not_obs = dsname in ['GISTEMP','HadCRUT5','BEST']
                
                 # If LENS ensemble
                if attrs[1] == 0:
                    # Extract mean & min and max of envelope
                    da_mean = ds[dsname+' mean']
                    da_std = ds[dsname].std('slice')*2
                    da_min = da_mean-da_std
                    da_max = da_mean+da_std
                
                    # Plot envelope
                    ax.fill_between(da_min[clist[1]].values, da_min.values,
                        da_max.values, color=attrs[2], alpha=0.1,
                        ec=None,label=ds_paper_names[dsname], zorder=attrs[4])
                    
                    # Plot mean
                    da_mean.plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4],label=ds_paper_names[dsname]+' mean')

                # If nudging ensemble
                elif attrs[1] == 1:
                    da = ds[dsname]
                    index = dict(ensemble_member=1)

                    # Plot first ensemble member
                    da.loc[index].plot(
                        ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4], label=ds_paper_names[dsname]) 
    
                    # Plot other ensemble members
                    for i in range(2,4):
                        index['ensemble_member'] = i
                
                        da.loc[index].plot(
                            ax=ax, color=attrs[2], x=clist[1], zorder=attrs[4], ls=attrs[3])  
                        
                # If observations
                elif attrs[1] == 3:
                    if not ((c == 0 and not_obs) or (c == 2 and var == 'MOC')):
                        if (dsname == 'NSIDC' or dsname == 'OSISAF'):
                            t_slice = dict(time=slice('1980-01-01','2024-12-31')) if c == 0 else dict(start_year=slice(1980,2005))
                            da = ds[dsname].loc[t_slice]
                        elif dsname == 'RAPID':
                            t_slice = dict(year=slice(2004,2024)) if c == 0 else dict(start_year=slice(2004,2005))
                            da = ds[dsname].loc[t_slice]
                        else:
                            da = ds[dsname]
                        da.plot(
                            ax=ax, color=attrs[2], ls=attrs[3], x=clist[1], zorder=attrs[4], label=ds_paper_names[dsname])

            # Formatting
            ax.set_title('')
            ax.set_ylabel(clist[3])
            ax.set_ylim(clist[4])
            p = 3*r+c
            ax.annotate(let_labels[p], (0.01,0.04), xycoords='axes fraction', fontweight='bold')
            if var == 'aice' and c == 0:
                ax.xaxis.set_major_locator(mdates.YearLocator(10,month=type_onemonth))
                ax.xaxis.set_minor_locator(mdates.YearLocator(5,month=type_onemonth))
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
                ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),
                            np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
            else:
                ax.set_xticks(ticks=xticks_loc, minor=False)
                ax.set_xticks(ticks=xticks_locmin, minor=True)
                ax.set_xlim(clist[6])
            ax.set_xlabel(clist[5])
            ax.grid(alpha=0.3,which='both')
            if c == 0:
                ax.set_yticks(ticks=yticksabs[var])

            if c == 1:
                # Legend data collection
                if r == 0:
                    h, l = ax.get_legend_handles_labels()
                    print(l)
                    hand_list += h[3:]
                    label_list += l[3:]
                    hand_list += h[0:3]
                    label_list += l[0:3]
                elif r != 0:
                    h, l = ax.get_legend_handles_labels()
                    hand_list += h[0:3]
                    label_list += l[0:3]

    # Create legend
    axleg = fig.add_subplot(gs[3,:])
    axleg.legend(hand_list, label_list, loc='upper center', ncol=4, frameon=False)
    axleg.axis('off')
    
    SaveFig(fig, graph_type_str, date_str, var, note)

    plt.close(fig)

['PI-control', 'PI-control mean', 'PI+winds', 'OBS (ERA5)', 'OBS (NSIDC)', 'OBS (OSISAF)']
CPU times: user 998 ms, sys: 11.2 ms, total: 1.01 s
Wall time: 1.01 s


In [53]:
%%time

if plots['comb'][0] and o_domain and var == 'MOC' and ens_type:

    lat_sel = [20, 26.5, 33, 40, 45, 55, slice(33,55)]
    
    row_dict = {0: ['PiC_UVnudge_LM2006', 'Blues'],
                1: ['GHG_UVnudge_LM', 'Greens'],
                2: ['HIST_UVnudge_LM', 'Reds']}
                # List: [0:dataset, 1:x dimension, 2:trend length, 3:y-label, 4:y-limit, 5:x-label, 6:x-limit]
    col_dict = {0: [[ds_abs, ds_abs_def], period, 1, ylabelabs[var], ylimabs[var], xlabelabs, xlimabs],
                1: [[ds_20trd, ds_20trd_def], xdim, 20, ylabeltwolinesingle[var], ylimtrd20[var], xlabeltrd(20), xlimtrd(20)],
                2: [[ds_40trd, ds_40trd_def], xdim, 40, ylabeltwolinesingle[var], ylimtrd40[var], xlabeltrd(40), xlimtrd(40)]}
    
    # Set up figure and subplots
    fig = plt.figure(figsize=(11,8),layout='constrained')
    gs = gridspec.GridSpec(hgt+1, wdth, height_ratios=[1,1,1,0.3],
                            width_ratios=[ywidth(1),ywidth(20),ywidth(40)],
                            wspace=0.4, hspace=0.6)
    hand_list = []
    label_list = []
    let_labels = FigLabelList(hgt*wdth)
    
    # Plot all data
    for r, dnames in row_dict.items():
        # Create axes for given row
        axa = fig.add_subplot(gs[r,0])
        ax20 = fig.add_subplot(gs[r,1])
        ax40 = fig.add_subplot(gs[r,2])
        axlist = [axa, ax20, ax40]
        dsname = dnames[0]
        cmap = mpl.colormaps[dnames[1]]

        # Take colors at regular i
        colors = cmap(np.linspace(0.3, 1, len(lat_sel)))

        for c, clist in col_dict.items():
            ax = axlist[c]
            ds = clist[0][0][dnames[0]]
            ds_def = clist[0][1][dnames[0]]
            attrs = ds_names[dsname]
            if c != 0:
                ax.axhline(0, color='k', zorder=0, lw=0.5)
                
            for i in range(len(lat_sel)):
                if i == len(lat_sel)-1:
                    da = ds_def
                    addon = ' 33-55N'
                    lw = 2
                else:
                    da = ds.sel(lat=lat_sel[i], method='nearest')
                    addon = ' '+str(lat_sel[i])+'N'
                    lw = 1

                # Plot ensemble mean
                da.plot(
                    ax=ax, color=colors[i], ls=attrs[3], x=clist[1], zorder=attrs[4],
                    label=ds_paper_names[dsname]+addon, lw=lw) 

                        
            # Formatting
            ax.set_title('')
            ax.set_ylabel(clist[3])
            ax.set_ylim(clist[4])
            p = 3*r+c
            ax.annotate(let_labels[p], (0,1.05), xycoords='axes fraction')
            if var == 'aice' and c == 0:
                ax.xaxis.set_major_locator(mdates.YearLocator(10,month=type_onemonth))
                ax.xaxis.set_minor_locator(mdates.YearLocator(5,month=type_onemonth))
                ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
                ax.set_xlim(np.datetime64(td_str+'-'+str(type_onemonth).zfill(2)+'-01'),
                            np.datetime64('2024-'+str(type_onemonth).zfill(2)+'-01'))
            else:
                ax.set_xticks(ticks=xticks_loc, minor=False)
                ax.set_xticks(ticks=xticks_locmin, minor=True)
                ax.set_xlim(clist[6])
            ax.set_xlabel(clist[5])
            ax.grid(alpha=0.3,which='both')
            if c == 0:
                ax.set_yticks(ticks=yticksabs[var])

        # Legend data collection
        h, l = ax.get_legend_handles_labels()
        hand_list += h[:]
        label_list += l[:]

    # Create legend
    axleg = fig.add_subplot(gs[3,:])
    axleg.legend(hand_list, label_list, loc='upper center', ncol=3, frameon=False)
    axleg.axis('off')
    
    SaveFig(fig, graph_type_str, date_str, var, note)

    plt.close(fig)

CPU times: user 26 µs, sys: 0 ns, total: 26 µs
Wall time: 8.82 µs
